<a href="https://colab.research.google.com/github/Sasindu99-ai/Statistical-Learning-e22445/blob/main/Assignments/Assignment%207c/e22445_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<pre style="color:green;">
Assignment 7    :   Bayesian Inference 7c
Course          :   ME2050
Reg. No.        :   E/22/445
Name            :   P.M.S.S. Wjethunga
</pre>

---

In [1]:
import numpy as np
import plotly.graph_objects as go
import scipy.stats as stats

	# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


1. Visualizing the MechanicsTo understand the dynamics of the 2PL model, we can visualize the probability of a correct response as a function of the user's ability $\theta$. The discrimination parameter $a_i$ affects the steepness of the curve, while the difficulty parameter $b_i$ shifts it horizontally.

In [2]:
# Define the 2PL probability function
def prob_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_vals = np.linspace(-4, 4, 400)

# Configurations: two distinct 'a' values; one 'a' paired with three 'b' values
configs = [
    {"a": 1.0, "b": 0.0, "name": "a=1.0, b=0.0 (Lower Discrimination)"},
    {"a": 2.5, "b": -1.0, "name": "a=2.5, b=-1.0 (High Discrim, Easy)"},
    {"a": 2.5, "b": 0.0, "name": "a=2.5, b=0.0 (High Discrim, Medium)"},
    {"a": 2.5, "b": 1.0, "name": "a=2.5, b=1.0 (High Discrim, Hard)"}
]

fig = go.Figure()

for config in configs:
    p_vals = prob_correct(theta_vals, config["a"], config["b"])
    fig.add_trace(go.Scatter(x=theta_vals, y=p_vals, mode='lines', name=config["name"]))

fig.update_layout(
    title="2PL Item Response Curves",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y=1 | θ)",
    template="plotly_white",
    legend=dict(x=0.02, y=0.98)
)

fig.show()

Interpretation: The parameter $b_i$ dictates the location where the item is most informative (where the probability of a correct response is $0.5$). Moving $b_i$ to the right (larger values) shifts the entire logistic curve horizontally to the right. This means a higher latent ability $\theta$ is required to achieve the same probability of answering the question correctly, reflecting a "harder" item.

2. Sequential Likelihood ContributionAssuming conditional independence of the item responses given $\Theta=\theta$, the likelihood function is constructed by the product of the probabilities of the observed outcomes.  For a single new response $y_k \in \{0, 1\}$ at step $k$, the likelihood contribution is a Bernoulli density parameterized by the 2PL curve:$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$The joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, \dots, y_k)$ is the product of the individual likelihoods up to step $k$:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

3. Mathematical Formulation of the Running UpdateBayesian inference updates the prior distribution into a posterior distribution after seeing the data. In a sequential setting, the posterior after $k-1$ items becomes the prior for the $k$-th item.  The recursive update up to a proportionality constant is written as:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$This states that the new running posterior is proportional to the likelihood of the $k$-th response multiplied by the posterior from step $k-1$.

4. Dynamic ShiftingIf a user answers a highly difficult item correctly ($y_k = 1$ and $b_k$ is large), the likelihood function evaluated at this step is simply $p_k(\theta)$. For a large $b_k$, $p_k(\theta)$ remains close to $0$ for small and average values of $\theta$, and only starts approaching $1$ for large values of $\theta$.Because the Bayesian update multiplies the prior by this likelihood, regions of $\theta$ that are small will be heavily penalized (multiplied by a number near $0$), while larger values of $\theta$ will be preserved. Consequently, normalizing this product shifts the peak (mode) of the running posterior density strongly to the right, signifying a sudden increase in the platform's estimate of the user's ability.

5. Tracking Certainty and SharpnessThe discrimination parameter $a_k$ controls the slope of the logistic curve at the point $\theta = b_k$.
- When $a_k$ is very large: The transition from a probability of $0$ to $1$ is extremely sharp. If the user answers such an item, the likelihood functions step sharply. Multiplying the running prior by a highly discriminative likelihood acts like a strict filter, rapidly cutting off one side of the distribution. This dramatically decreases the variance of the posterior, increasing the "sharpness" and certainty of the estimate.
- When $a_k$ is very small: The logistic curve is flat. The likelihood evaluates to roughly $0.5$ across a wide range of $\theta$. Multiplying the prior by a relatively flat function does very little to change its shape. The posterior variance remains largely unchanged, reflecting that a poorly discriminating item provides little new information.

6. Numerical Implementation of a Running GridTo approximate this continuous update algorithmically on a computer without relying on complex analytical integrals (such as Markov chain Monte Carlo methods):
    1. Grid Initialization: Define a discrete, evenly spaced array of $\theta$ values spanning a plausible range (e.g., $M$ points between $-4$ and $4$). Let the grid spacing be $\Delta \theta$.
    2. Prior Initialization: Evaluate the standard normal PDF across the grid. Normalize the array so it sums to $1$ when integrated (i.e., array sum $\times \Delta \theta = 1$).
    3. Sequential Update: For each new item $k$:
        - Evaluate the likelihood $L(y_k \mid \theta)$ across the entire grid of $\theta$ values.
        - Perform an element-wise multiplication of the likelihood array and the current posterior array.
    4. Sequential Normalization: Calculate the area under the new unnormalized curve by summing all elements and multiplying by $\Delta \theta$. Divide the unnormalized array by this area constant to produce a valid probability density function.
    5. Repeat: Use this normalized array as the prior for step $k+1$.

7. Evaluating Convergence over the Timeline
The code below implements the simulation and tracking of the Bayesian point estimates. It tracks the posterior mean (Bayes estimate) and the Maximum A Posteriori (MAP) estimate.

In [3]:
# Parameters
np.random.seed(42)
n_items = 20
theta_true = 0.75
grid_size = 1000
theta_grid = np.linspace(-4, 4, grid_size)
delta_theta = theta_grid[1] - theta_grid[0]

# Initialize prior
posterior = stats.norm.pdf(theta_grid, 0, 1)
posterior /= np.sum(posterior) * delta_theta

bayes_estimates = [np.sum(theta_grid * posterior) * delta_theta]
map_estimates = [theta_grid[np.argmax(posterior)]]

# Item parameters
a_params = np.random.uniform(0.5, 2.0, n_items)
b_params = np.random.normal(0, 1, n_items)

# Simulation loop
for k in range(n_items):
    # Simulate user response
    a_k = a_params[k]
    b_k = b_params[k]
    p_correct_true = 1 / (1 + np.exp(-a_k * (theta_true - b_k)))
    y_k = 1 if np.random.uniform(0, 1) < p_correct_true else 0

    # Calculate likelihood on grid
    p_grid = 1 / (1 + np.exp(-a_k * (theta_grid - b_k)))
    likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))

    # Update posterior
    unnormalized_posterior = likelihood * posterior
    posterior = unnormalized_posterior / (np.sum(unnormalized_posterior) * delta_theta)

    # Calculate estimates[cite: 1]
    bayes_est = np.sum(theta_grid * posterior) * delta_theta
    map_est = theta_grid[np.argmax(posterior)]

    bayes_estimates.append(bayes_est)
    map_estimates.append(map_est)

# Plotting
steps = np.arange(n_items + 1)
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines+markers', name='Bayes Estimate (Posterior Mean)'))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate (Posterior Mode)'))
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True θ = 0.75", annotation_position="top right")

fig.update_layout(
    title="Convergence of Running Ability Estimators",
    xaxis_title="Item Number (k)",
    yaxis_title="Estimated Latent Ability (θ)",
    template="plotly_white",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    legend=dict(x=0.02, y=0.02)
)

fig.show()

Analysis:
As $k$ increases, the distance between both estimators ($\widehat{\theta}_{\mathrm{Bayes}}$ and $\widehat{\theta}_{\mathrm{MAP}}$) and the true latent ability $\theta_{\text{true}}$ generally decreases. The lines oscillate heavily in the early steps because the prior is still highly influential and single items cause large relative shifts. As more data is gathered, the likelihood dominates the standard normal prior, the variance shrinks, and the estimates converge toward the true value. This implies that as the user answers more questions, the platform's confidence in its measurement grows, making the estimates highly robust against single anomalous responses.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

1. Structural Probability and Properties
The Beta distribution is an ideal prior for probabilities because its domain is strictly bounded between 0 and 1.

In [4]:
theta_vals = np.linspace(0, 1, 500)

configs = [
    {"alpha": 1, "beta": 1, "name": "Beta(1, 1) - Uninformative"},
    {"alpha": 2, "beta": 8, "name": "Beta(2, 8) - Right-skewed (Peak near 0)"},
    {"alpha": 8, "beta": 2, "name": "Beta(8, 2) - Left-skewed (Peak near 1)"}
]

fig = go.Figure()

for config in configs:
    pdf_vals = stats.beta.pdf(theta_vals, config["alpha"], config["beta"])
    fig.add_trace(go.Scatter(x=theta_vals, y=pdf_vals, mode='lines', name=config["name"]))

fig.update_layout(
    title="Beta Distribution Shapes for CTR Prior",
    xaxis_title="Conversion Rate (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    legend=dict(x=0.02, y=0.98)
)

fig.show()

Interpretation:
- $\text{Beta}(1, 1)$ represents a completely uniform state of belief; every conversion rate is equally likely before any data is seen.
- $\text{Beta}(2, 8)$ shifts the center of mass toward 0. This models a pessimistic (but realistic) scenario where we expect the CTR to be low.
- $\text{Beta}(8, 2)$ shifts the mass toward 1, reflecting a strong prior belief that the advertisement will perform exceptionally well.

2. Sequential Likelihood and Joint HistoryConditional on $\Theta = \theta$, a single user impression is a Bernoulli trial. The likelihood contribution of a single observation $y_k$ is:$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$Because each impression is assumed independent, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual likelihoods:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

3. Closed-Form Analytical Updates (Conjugacy)Bayesian inference updates the prior distribution of $\Theta$ into a posterior distribution after seeing the data. Using Bayes' Theorem sequentially, the new posterior is proportional to the new likelihood multiplied by the previous step's posterior:  $$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Substituting the likelihood and the Beta prior density from step $k-1$:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \times \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$Combining the exponents algebraically yields:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$This resulting expression is exactly the functional form of a new Beta distribution. Therefore, the distribution remains in the Beta family with the following simple arithmetic updates:  $\alpha_k = \alpha_{k-1} + y_k$$\beta_k = \beta_{k-1} + (1 - y_k)$The running Posterior Mean is simply the expected value of this updated Beta distribution:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

4. Dynamic Shifting Mechanics
    1. Observed Click ($y_k = 1$): The success parameter $\alpha$ increases by 1, while $\beta$ remains unchanged. This pulls the peak (and the center of mass) of the Beta distribution to the right, increasing the estimated CTR.
    2. Observed Non-click ($y_k = 0$): The failure parameter $\beta$ increases by 1, while $\alpha$ remains unchanged. This pushes the peak of the Beta distribution to the left, decreasing the estimated CTR.

- Contrast with Non-Conjugate Models:
In the Beta-Binomial setup, the functional form of the prior perfectly matches the likelihood, allowing the exact posterior to be derived through simple addition. In non-conjugate models (like the 2PL IRT model with a Normal prior and Logistic likelihood), multiplying the two functions does not yield a recognizable, standard distribution. Consequently, those models require computationally expensive numerical grid approximations or MCMC sampling to normalize the posterior at every step.

5. Running Point Estimators

We can extract point estimates directly from the updated shape parameters without any integration.



| Estimator Type | Closed-Form Equation | Condition |
|---|---|---|
| **Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) | $\frac{\alpha_k}{\alpha_k + \beta_k}$ | Always valid for $\alpha_k, \beta_k > 0$ |
| **Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) | $\frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$ | Requires $\alpha_k > 1$ and $\beta_k > 1$ |

6. Performance Tracking and Convergence Analysis

In [5]:
# Setup
np.random.seed(101)
n_impressions = 100
theta_true = 0.35

# Initialize prior Beta(1, 1)
alpha_k = 1
beta_k = 1

bayes_estimates = []
map_estimates = []

# Simulation loop
for k in range(1, n_impressions + 1):
    # Simulate user interaction
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # Analytical update (Conjugacy)
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Calculate point estimators
    bayes_est = alpha_k / (alpha_k + beta_k)

    # MAP is only defined if alpha, beta > 1. Fallback to mean if undefined.
    if alpha_k > 1 and beta_k > 1:
        map_est = (alpha_k - 1) / (alpha_k + beta_k - 2)
    else:
        map_est = bayes_est

    bayes_estimates.append(bayes_est)
    map_estimates.append(map_est)

# Plotting
steps = np.arange(1, n_impressions + 1)
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines', name='Bayes Estimate (Mean)'))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines', name='MAP Estimate (Mode)'))
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True θ = 0.35", annotation_position="top right")

fig.update_layout(
    title="Sequential Beta-Binomial Updates for CTR Tracking",
    xaxis_title="Impression Number (k)",
    yaxis_title="Estimated CTR (θ)",
    template="plotly_white",
    legend=dict(x=0.6, y=0.1)
)

fig.show()

Analysis:
At the beginning of the timeline, the estimates oscillate dramatically because the uniform prior ($\alpha=1, \beta=1$) exerts very little influence, making the estimators highly sensitive to single clicks or non-clicks. As $k$ approaches 100, the distance between both estimators and $\theta_{\text{true}}$ steadily shrinks, and the lines flatten out. This implies that as evidence (data) accumulates, it overwhelmingly "washes out" the choice of the initial prior, funneling the model's posterior belief strictly toward the true underlying conversion rate.